# Pattern 形态可视化教学

这个 notebook 用模拟数据把 `docs/pattern_backtest_score_guide.md` 里提到的常见结构形态、`HH / HL / LH / LL` 结构信号和 K 线形态画出来，目标是帮助你更直观地认识“它长什么样、通常代表什么含义”。

说明：
- 这里的图都是教学示意图，不代表真实市场噪声。
- 它更接近本项目“程序化定义下的结构信号”，不是严格教材里的唯一标准画法。
- 颜色约定：绿色偏多，红色偏空，灰色偏中性或观察。

In [1]:
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

pd.set_option('display.max_colwidth', None)

# 颜色只服务于教学识别
BULL_COLOR = '#2ca02c'
BEAR_COLOR = '#d62728'
NEUTRAL_COLOR = '#7f7f7f'


def bias_color(label: str) -> str:
    return {
        '偏多': BULL_COLOR,
        '偏空': BEAR_COLOR,
    }.get(label, NEUTRAL_COLOR)


def subplot_position(index: int, cols: int = 3) -> tuple[int, int]:
    return index // cols + 1, index % cols + 1


# 把简短 OHLC 序列转成 plotly 需要的表结构
def make_ohlc_frame(ohlc: list[tuple[float, float, float, float]]) -> pd.DataFrame:
    frame = pd.DataFrame(ohlc, columns=['Open', 'High', 'Low', 'Close'])
    frame['x'] = list(range(1, len(frame) + 1))
    return frame

## 1. 结构形态

这些形态对应 guide 里的字符串列，比如 `triangle_pattern`、`wedge_pattern`、`double_pattern`、`head_shoulder_pattern`、`multiple_top_bottom_pattern`、`channel_pattern`。

这里用“模拟拐点路径”强调形状本身，方便先记住轮廓。注意：`Wedge Up` 和 `Wedge Down` 更依赖上下文，它们更像收敛结构，不建议脱离动量或趋势指标单独解读。

In [2]:
# 这里用模拟拐点序列强调“形状”，不追求真实行情噪声
structure_patterns = [
    {'column': 'triangle_pattern', 'value': 'Ascending Triangle', 'meaning': '高点平台、低点逐步抬高', 'intuition': '偏多', 'points': [100, 104, 101.5, 104, 103, 104, 104.2, 107]},
    {'column': 'triangle_pattern', 'value': 'Descending Triangle', 'meaning': '低点平台、高点逐步下移', 'intuition': '偏空', 'points': [104, 100, 103, 100, 102, 100, 99, 96.5]},
    {'column': 'wedge_pattern', 'value': 'Wedge Up', 'meaning': '高低点都上移，但波动逐步收敛', 'intuition': '观察', 'points': [100, 103.8, 101.8, 104.8, 103.3, 105.4, 104.6, 105.8]},
    {'column': 'wedge_pattern', 'value': 'Wedge Down', 'meaning': '高低点都下移，但波动逐步收敛', 'intuition': '观察', 'points': [106, 102.5, 104.6, 101.8, 103.7, 101.0, 102.9, 100.5]},
    {'column': 'double_pattern', 'value': 'Double Top', 'meaning': '两次上冲高位失败', 'intuition': '偏空', 'points': [100, 104.5, 101.5, 104.2, 101.0, 98.5]},
    {'column': 'double_pattern', 'value': 'Double Bottom', 'meaning': '两次下探低位后企稳', 'intuition': '偏多', 'points': [104, 99.8, 102.5, 100.0, 103.5, 106.5]},
    {'column': 'head_shoulder_pattern', 'value': 'Head and Shoulder', 'meaning': '头部高于两侧肩部', 'intuition': '偏空', 'points': [100, 104, 102, 106.5, 102, 104, 99.5]},
    {'column': 'head_shoulder_pattern', 'value': 'Inverse Head and Shoulder', 'meaning': '中间低点低于两侧肩部', 'intuition': '偏多', 'points': [106, 102.5, 104, 99.5, 104, 102.5, 107]},
    {'column': 'multiple_top_bottom_pattern', 'value': 'Multiple Top', 'meaning': '多次上冲阻力位失败', 'intuition': '偏空', 'points': [100, 104.2, 101.5, 104.0, 102.0, 104.1, 101.2]},
    {'column': 'multiple_top_bottom_pattern', 'value': 'Multiple Bottom', 'meaning': '多次下探支撑位后企稳', 'intuition': '偏多', 'points': [104, 100.0, 102.2, 99.8, 101.8, 100.1, 103.2]},
    {'column': 'channel_pattern', 'value': 'Channel Up', 'meaning': '价格在向上通道中运行', 'intuition': '偏多', 'points': [100, 102.2, 101.0, 103.1, 101.9, 104.0, 102.9, 105.0]},
    {'column': 'channel_pattern', 'value': 'Channel Down', 'meaning': '价格在向下通道中运行', 'intuition': '偏空', 'points': [105, 102.8, 104.0, 101.8, 103.0, 100.8, 102.0, 99.8]},
]

structure_table = pd.DataFrame([
    {
        '列名': item['column'],
        '取值': item['value'],
        '大致含义': item['meaning'],
        '常见直觉': item['intuition'],
        'score_value_map 示例': f"{{'{item['column']}': '{item['value']}'}}",
    }
    for item in structure_patterns
])

display(structure_table)

fig = make_subplots(
    rows=4,
    cols=3,
    subplot_titles=[item['value'] for item in structure_patterns],
    vertical_spacing=0.08,
)

for idx, item in enumerate(structure_patterns):
    row, col = subplot_position(idx, cols=3)
    x = list(range(1, len(item['points']) + 1))
    fig.add_trace(
        go.Scatter(
            x=x,
            y=item['points'],
            mode='lines+markers',
            line={'color': bias_color(item['intuition']), 'width': 3, 'shape': 'spline'},
            marker={'size': 7},
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    fig.update_xaxes(showticklabels=False, row=row, col=col)
    fig.update_yaxes(showticklabels=False, row=row, col=col)

fig.update_layout(
    title='常见结构形态示意图',
    template='plotly_white',
    height=1100,
    width=1100,
    margin={'t': 80, 'l': 20, 'r': 20, 'b': 20},
)
fig.show()

,列名,取值,大致含义,常见直觉,score_value_map 示例
0,triangle_pattern,Ascending Triangle,高点平台、低点逐步抬高,偏多,{'triangle_pattern': 'Ascending Triangle'}
1,triangle_pattern,Descending Triangle,低点平台、高点逐步下移,偏空,{'triangle_pattern': 'Descending Triangle'}
2,wedge_pattern,Wedge Up,高低点都上移，但波动逐步收敛,观察,{'wedge_pattern': 'Wedge Up'}
3,wedge_pattern,Wedge Down,高低点都下移，但波动逐步收敛,观察,{'wedge_pattern': 'Wedge Down'}
4,double_pattern,Double Top,两次上冲高位失败,偏空,{'double_pattern': 'Double Top'}
5,double_pattern,Double Bottom,两次下探低位后企稳,偏多,{'double_pattern': 'Double Bottom'}
6,head_shoulder_pattern,Head and Shoulder,头部高于两侧肩部,偏空,{'head_shoulder_pattern': 'Head and Shoulder'}
7,head_shoulder_pattern,Inverse Head and Shoulder,中间低点低于两侧肩部,偏多,{'head_shoulder_pattern': 'Inverse Head and Shoulder'}
8,multiple_top_bottom_pattern,Multiple Top,多次上冲阻力位失败,偏空,{'multiple_top_bottom_pattern': 'Multiple Top'}
9,multiple_top_bottom_pattern,Multiple Bottom,多次下探支撑位后企稳,偏多,{'multiple_top_bottom_pattern': 'Multiple Bottom'}


## 2. `signal` 的 `HH / HL / LH / LL`

这四个值不是完整大形态，而是市场结构语言：
- `HH`：Higher High，更高的高点。
- `HL`：Higher Low，更高的低点。
- `LH`：Lower High，更低的高点。
- `LL`：Lower Low，更低的低点。

In [3]:
signal_examples = [
    {'value': 'HH', 'meaning': '更高的高点，趋势强化', 'points': [100, 104, 102, 106, 103.5, 108], 'markers': [1, 3, 5], 'intuition': '偏多', 'textposition': 'top center'},
    {'value': 'HL', 'meaning': '更高的低点，结构转强', 'points': [106, 100, 107, 102, 108, 104], 'markers': [1, 3, 5], 'intuition': '偏多', 'textposition': 'bottom center'},
    {'value': 'LH', 'meaning': '更低的高点，反弹转弱', 'points': [100, 106, 102.5, 104.5, 101.8, 103], 'markers': [1, 3, 5], 'intuition': '偏空', 'textposition': 'top center'},
    {'value': 'LL', 'meaning': '更低的低点，下跌延续', 'points': [106, 102, 105, 99.5, 103, 97], 'markers': [1, 3, 5], 'intuition': '偏空', 'textposition': 'bottom center'},
]

signal_table = pd.DataFrame([
    {
        '取值': item['value'],
        '大致含义': item['meaning'],
        '常见直觉': item['intuition'],
        'score_value_map 示例': "{'signal': ['HH', 'HL']}" if item['value'] in {'HH', 'HL'} else "{'signal': ['LH', 'LL']}",
    }
    for item in signal_examples
])

display(signal_table)

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[f"{item['value']} | {item['meaning']}" for item in signal_examples],
    vertical_spacing=0.14,
)

for idx, item in enumerate(signal_examples):
    row, col = subplot_position(idx, cols=2)
    x = list(range(1, len(item['points']) + 1))
    fig.add_trace(
        go.Scatter(
            x=x,
            y=item['points'],
            mode='lines+markers',
            line={'color': bias_color(item['intuition']), 'width': 3},
            marker={'size': 7},
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=[x[i] for i in item['markers']],
            y=[item['points'][i] for i in item['markers']],
            mode='markers+text',
            text=[item['value']] * len(item['markers']),
            textposition=item['textposition'],
            marker={'size': 10, 'color': bias_color(item['intuition'])},
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    fig.update_xaxes(showticklabels=False, row=row, col=col)
    fig.update_yaxes(showticklabels=False, row=row, col=col)

fig.update_layout(
    title='HH / HL / LH / LL 市场结构示意图',
    template='plotly_white',
    height=700,
    width=980,
    margin={'t': 80, 'l': 20, 'r': 20, 'b': 20},
)
fig.show()

,取值,大致含义,常见直觉,score_value_map 示例
0,HH,更高的高点，趋势强化,偏多,"{'signal': ['HH', 'HL']}"
1,HL,更高的低点，结构转强,偏多,"{'signal': ['HH', 'HL']}"
2,LH,更低的高点，反弹转弱,偏空,"{'signal': ['LH', 'LL']}"
3,LL,更低的低点，下跌延续,偏空,"{'signal': ['LH', 'LL']}"


## 3. K 线形态

这些形态对应 guide 里的布尔列，比如 `BullishEngulfing`、`BearishEngulfing`、`Doji`、`Hammer`、`MorningStar` 等。

下面用很短的模拟 OHLC 序列，把每个形态最关键的几根 K 线画出来，方便你观察实体、上影线、下影线和吞没关系。

In [4]:
# 每个形态只保留最关键的几根 K 线，便于教学观察
candle_patterns = [
    {
        'column': 'Doji',
        'meaning': '开收接近，多空犹豫',
        'intuition': '观察',
        'ohlc': [(102.0, 103.0, 101.0, 102.8), (102.8, 103.5, 101.9, 103.2), (103.1, 105.0, 101.2, 103.12), (103.0, 103.8, 101.8, 102.6)],
    },
    {
        'column': 'Hammer',
        'meaning': '下跌后长下影，小实体，常见止跌信号',
        'intuition': '偏多',
        'ohlc': [(106.0, 106.5, 104.8, 105.0), (105.0, 105.2, 102.8, 103.4), (103.2, 103.8, 99.4, 103.6), (103.7, 105.8, 103.5, 105.2)],
    },
    {
        'column': 'HangingMan',
        'meaning': '上涨后长下影，小实体，常见转弱提醒',
        'intuition': '偏空',
        'ohlc': [(100.0, 101.6, 99.8, 101.2), (101.2, 103.8, 101.0, 103.4), (103.5, 103.9, 100.1, 103.2), (103.0, 103.2, 101.2, 101.6)],
    },
    {
        'column': 'InvertedHammer',
        'meaning': '下跌后长上影，小实体，可能尝试反弹',
        'intuition': '偏多',
        'ohlc': [(106.0, 106.3, 104.8, 105.0), (105.0, 105.1, 102.9, 103.5), (103.4, 106.8, 103.1, 103.6), (103.8, 105.6, 103.5, 105.0)],
    },
    {
        'column': 'ShootingStar',
        'meaning': '上涨后长上影，小实体，常见冲高回落',
        'intuition': '偏空',
        'ohlc': [(100.0, 101.8, 99.9, 101.5), (101.5, 104.0, 101.2, 103.6), (103.7, 107.2, 103.4, 103.5), (103.4, 103.6, 101.0, 101.6)],
    },
    {
        'column': 'BullishEngulfing',
        'meaning': '大阳吞没前一根阴线，常见偏多反转',
        'intuition': '偏多',
        'ohlc': [(106.0, 106.4, 104.8, 105.0), (105.0, 105.2, 103.6, 104.0), (103.7, 106.2, 103.4, 105.8), (105.9, 106.8, 105.5, 106.4)],
    },
    {
        'column': 'BearishEngulfing',
        'meaning': '大阴吞没前一根阳线，常见偏空反转',
        'intuition': '偏空',
        'ohlc': [(100.0, 101.2, 99.8, 100.6), (100.6, 103.0, 100.4, 102.4), (102.7, 102.9, 99.7, 100.1), (100.0, 100.4, 98.8, 99.2)],
    },
    {
        'column': 'MorningStar',
        'meaning': '先跌、再稳、再反弹的三根组合',
        'intuition': '偏多',
        'ohlc': [(106.0, 106.2, 102.8, 103.2), (102.9, 103.4, 102.1, 102.7), (102.8, 105.8, 102.6, 105.2), (105.2, 106.0, 104.8, 105.8)],
    },
    {
        'column': 'BullishHarami',
        'meaning': '大阴后出现被包住的小实体，常见早期止跌信号',
        'intuition': '偏多',
        'ohlc': [(107.0, 107.5, 105.8, 106.2), (106.0, 106.3, 102.5, 103.0), (103.4, 104.1, 103.1, 103.8), (103.9, 105.4, 103.6, 104.9)],
    },
    {
        'column': 'BearishHarami',
        'meaning': '大阳后出现被包住的小实体，常见早期转弱信号',
        'intuition': '偏空',
        'ohlc': [(100.0, 101.4, 99.8, 101.2), (101.2, 105.8, 101.0, 105.0), (104.6, 104.9, 104.0, 104.3), (104.0, 104.2, 102.0, 102.6)],
    },
    {
        'column': 'PiercingPattern',
        'meaning': '大阴后强反弹，收回前阴线一半以上',
        'intuition': '偏多',
        'ohlc': [(107.0, 107.2, 106.0, 106.5), (106.0, 106.2, 102.4, 103.0), (102.3, 105.0, 101.8, 104.8), (104.9, 106.0, 104.5, 105.6)],
    },
    {
        'column': 'DarkCloudCover',
        'meaning': '大阳后明显回落，跌回前阳线实体一半以下',
        'intuition': '偏空',
        'ohlc': [(99.2, 100.6, 99.0, 100.0), (100.0, 104.2, 99.8, 103.8), (104.4, 104.8, 101.0, 101.5), (101.4, 101.8, 99.8, 100.2)],
    },
]

candle_table = pd.DataFrame([
    {
        '列名': item['column'],
        '典型含义': item['meaning'],
        '常见直觉': item['intuition'],
    }
    for item in candle_patterns
])

display(candle_table)

fig = make_subplots(
    rows=4,
    cols=3,
    subplot_titles=[item['column'] for item in candle_patterns],
    vertical_spacing=0.08,
)

for idx, item in enumerate(candle_patterns):
    row, col = subplot_position(idx, cols=3)
    frame = make_ohlc_frame(item['ohlc'])
    fig.add_trace(
        go.Candlestick(
            x=frame['x'],
            open=frame['Open'],
            high=frame['High'],
            low=frame['Low'],
            close=frame['Close'],
            increasing_line_color=BULL_COLOR,
            decreasing_line_color=BEAR_COLOR,
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=frame['x'],
            y=frame['Close'],
            mode='lines',
            line={'color': bias_color(item['intuition']), 'width': 1.5, 'dash': 'dot'},
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    fig.update_xaxes(showticklabels=False, row=row, col=col)
    fig.update_yaxes(showticklabels=False, row=row, col=col)

for axis_idx in range(1, len(candle_patterns) + 1):
    axis_name = 'xaxis' if axis_idx == 1 else f'xaxis{axis_idx}'
    fig.layout[axis_name].rangeslider = {'visible': False}

fig.update_layout(
    title='常见 K 线形态示意图',
    template='plotly_white',
    height=1200,
    width=1100,
    margin={'t': 80, 'l': 20, 'r': 20, 'b': 20},
)
fig.show()

,列名,典型含义,常见直觉
0,Doji,开收接近，多空犹豫,观察
1,Hammer,下跌后长下影，小实体，常见止跌信号,偏多
2,HangingMan,上涨后长下影，小实体，常见转弱提醒,偏空
3,InvertedHammer,下跌后长上影，小实体，可能尝试反弹,偏多
4,ShootingStar,上涨后长上影，小实体，常见冲高回落,偏空
5,BullishEngulfing,大阳吞没前一根阴线，常见偏多反转,偏多
6,BearishEngulfing,大阴吞没前一根阳线，常见偏空反转,偏空
7,MorningStar,先跌、再稳、再反弹的三根组合,偏多
8,BullishHarami,大阴后出现被包住的小实体，常见早期止跌信号,偏多
9,BearishHarami,大阳后出现被包住的小实体，常见早期转弱信号,偏空


## 4. 如何把这些图和 guide 对起来

如果你以后把这些形态放回回测逻辑里，可以直接对应 guide 里的 `score_value_map` 思路，比如：

```python
score_value_map = {
    'triangle_pattern': 'Ascending Triangle',
    'double_pattern': 'Double Bottom',
    'channel_pattern': 'Channel Up',
    'signal': ['HH', 'HL'],
}
```

几个记忆点：
- `Ascending Triangle`、`Double Bottom`、`Inverse Head and Shoulder`、`Channel Up`、`HH / HL` 常被当成偏多结构。
- `Descending Triangle`、`Double Top`、`Head and Shoulder`、`Channel Down`、`LH / LL` 常被当成偏空结构。
- `Wedge Up`、`Wedge Down` 更像“收敛结构”，方向最好结合 `momentum_*`、`MACDh_*`、`rsi_*` 一起确认。
- K 线形态更适合当“加分项”或辅助确认，不建议单独作为唯一主 score。